In [ ]:
pip install langchain langchain-community langchain-core langchain-openai langgraph

In [ ]:
import os
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import END, StateGraph

# -------------------------------------------------------------
# 1. Environment Configuration
# -------------------------------------------------------------
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
search_tool = DuckDuckGoSearchRun()

# -------------------------------------------------------------
# 2. State Management (Shared Global State)
# -------------------------------------------------------------
class TravelState(TypedDict):
    destination: str
    days: int
    max_budget_usd: int
    itinerary_draft: str
    compliance_feedback: str
    is_valid: bool
    loop_count: int

# -------------------------------------------------------------
# 3. Agent Node Definitions
# -------------------------------------------------------------

def planner_node(state: TravelState):
    print(f"\n=== [AGENT] Travel Planner: Designing itinerary for {state['destination']} ===")
    dest = state["destination"]
    days = state["days"]
    budget = state["max_budget_usd"]
    feedback = state.get("compliance_feedback", "")
    
    # Live Search to pull up-to-date landmark and activity prices
    search_query = f"top activities landmarks ticket prices {dest} 2026"
    print(f"Executing Tool Search: '{search_query}'")
    search_results = search_tool.run(search_query)
    
    base_prompt = f"""You are an elite travel concierge. Draft a realistic day-by-day vacation itinerary for {days} days in {dest}.
    The user's maximum total budget is ${budget} USD.
    
    Use these live search insights to stay realistic with attractions and costs:\n{search_results}
    
    Include a 'Estimated Cost Breakdown' section at the end listing explicit dollar values for lodging, food, and activities."""
    
    # If the budget checker rejected it, inject feedback to modify the draft
    if feedback:
        print(f"Applying Compliance Feedback: {feedback}")
        base_prompt += f"\n\nCRITICAL MODIFICATIONS REQUIRED:\n{feedback}\nAdjust the activities or lodging selections to strictly meet the guidelines."

    response = llm.invoke([HumanMessage(content=base_prompt)])
    return {"itinerary_draft": response.content}


def compliance_node(state: TravelState):
    print("\n=== [FAILURE HANDLING] Compliance Auditor: Validating budget and structure ===")
    draft = state["itinerary_draft"]
    max_budget = state["max_budget_usd"]
    current_loops = state.get("loop_count", 0)
    
    # Automated safety failure check: Itineraries must include explicit financial breakdowns
    if "Cost" not in draft and "Estimated" not in draft:
        feedback = "Rejected: The itinerary is missing an explicit dollar cost breakdown. Rewrite it ensuring itemized costs are added."
        print("[REJECTED] Missing financial estimates. Triggering validation loop.")
        return {"compliance_feedback": feedback, "is_valid": False, "loop_count": current_loops + 1}
        
    # Programmatic check to simulate budget constraints on the first pass
    if current_loops == 0:
        feedback = f"Rejected: The initial plan slightly exceeds the maximum allowed cap of ${max_budget}. Swap expensive excursions out for highly rated free or low-cost alternatives."
        print(f"[REJECTED] Simulated budget overage constraint triggered.")
        return {"compliance_feedback": feedback, "is_valid": False, "loop_count": current_loops + 1}
        
    print("[APPROVED] Itinerary meets all financial constraints and layout rules.")
    return {"compliance_feedback": "APPROVED", "is_valid": True, "loop_count": current_loops + 1}

# -------------------------------------------------------------
# 4. Conditional Edge Logic
# -------------------------------------------------------------
def route_after_audit(state: TravelState):
    if state["is_valid"]:
        return "end"
    return "planner"

# -------------------------------------------------------------
# 5. Graph Assembly & Compilation
# -------------------------------------------------------------
builder = StateGraph(TravelState)

# Define architecture nodes
builder.add_node("planner", planner_node)
builder.add_node("compliance", compliance_node)

# Link sequential pipeline flow
builder.set_entry_point("planner")
builder.add_edge("planner", "compliance")

# Add the routing edge conditional logic
builder.add_conditional_edges(
    "compliance",
    route_after_audit,
    {
        "planner": "planner",
        "end": END
    }
)

app = builder.compile()

# -------------------------------------------------------------
# 6. Runtime Execution Block
# -------------------------------------------------------------
if __name__ == "__main__":
    travel_input = {
        "destination": "Tokyo, Japan",
        "days": 4,
        "max_budget_usd": 1200,
        "itinerary_draft": "",
        "compliance_feedback": "",
        "is_valid": False,
        "loop_count": 0
    }
    
    print("Initiating Multi-Agent Travel Coordinator Workflow...")
    final_state = app.invoke(travel_input)
    
    print("\n" + "="*80)
    print("FINAL VALIDATED TRAVEL ITINERARY")
    print("="*80)
    print(final_state["itinerary_draft"])